In [19]:
import base64
import mimetypes
from pathlib import Path
from IPython.display import Markdown, display

VISION_MODEL = "qwen/qwen3.8-27b"


def encode_image(image_path):
    """Read an image file and return it as a base64 data URL."""
    path = Path(image_path)
    if not path.exists():
        raise FileNotFoundError(f"Image not found: {path}")

    mime_type = mimetypes.guess_type(path)[0] or "image/png"   # handles png, jpg, webp
    image_b64 = base64.b64encode(path.read_bytes()).decode("utf-8")
    return f"data:{mime_type};base64,{image_b64}"


def ask_about_image(prompt, image_path, model=VISION_MODEL, max_tokens=1000):
    # Fire risk assessment prompt
    prompt = """
                Analyze the attached satellite image of a property with these specific steps:

                1. Residence identification: Locate the primary residence on the property by looking for:
                - The largest roofed structure 
                - Typical residential features (driveway connection, regular geometry)
                - Distinction from other structures (garages, sheds, pools)
                Describe the residence's location relative to property boundaries and other features.

                2. Tree overhang analysis: Examine all trees near the primary residence:
                - Identify any trees whose canopy extends directly over any portion of the roof
                - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75-100%)
                - Note particularly dense areas of overhang

                3. Fire risk assessment: For any overhanging trees, evaluate:
                - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
                - Proximity to chimneys, vents, or other roof openings if visible
                - Areas where branches create a "bridge" between wildland vegetation and the structure
                
                4. Defensible space identification: Assess the property's overall vegetative structure:
                - Identify if trees connect to form a continuous canopy over or near the home
                - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

                5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
                - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space around the structure
                - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between tree canopies
                - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected tree canopies, multiple points of vulnerability
                - Rating 4 (Severe Risk): Extensive overhang (>50% of roof), dense vegetation against structure, numerous ember catch points, limited defensible space

                For each item above (1-5), write one sentence summarizing your findings, with your final response being the numeric Fire Risk Rating (1-4) with a brief justification.
            """



    """Send a text prompt plus one image, return the model's text answer."""
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt},
                        {"type": "image_url", "image_url": {"url": encode_image(image_path)}},
                    ],
                }
            ],
            max_tokens=max_tokens,
        )
        return response.choices[0].message.content or "No response text returned."
    except Exception as e:
        return f"Request failed: {e}"

In [20]:
image_path = Path("Image_Support") / "images" / "prop7.png"

answer = ask_about_image("Describe this property and list any visible issues.", image_path)
display(Markdown(answer))

Based on the satellite imagery provided, here is the analysis of the property:

1.  **Residence identification:** The primary residence is an L-shaped structure with a light gray hip roof, featuring a central chimney, situated centrally within a dense forest environment.
2.  **Tree overhang analysis:** The tree canopy is extremely dense and directly overhangs the roof on at least three sides (top, right, and bottom), covering approximately 50-75% of the total roof surface area.
3.  **Fire risk assessment:** Vegetation creates a significant "ladder fuel" scenario with branches directly touching the eaves and roof deck, providing an immediate conduit for external fire to reach the structure and catch in the roof peak.
4.  **Defensible space identification:** There is virtually no defensible space; the home is completely enveloped by a continuous canopy of mature trees and heavy ground cover, eliminating the buffer required to slow fire spread.
5.  **Fire risk rating:** Based on the extensive overhang and dense vegetation directly against the structure, the Fire Risk Rating is 4 (Severe Risk).

**4**

**Justification:** The residence is completely surrounded by dense forest vegetation with significant tree overhang covering a large portion of the roof, offering no defensible space to mitigate fire spread.

    Image_Support\images\prop1.png
    Image_Support\images\prop2.png
    Image_Support\images\prop3.png
    Image_Support\images\prop4.png
    Image_Support\images\prop5.png
    Image_Support\images\prop6.png
    Image_Support\images\prop7.png